## Init config

In [1]:
import torch
from common_functions_python import set_config_file, test_function
config_file = {
                'name': 'DINO_features',
                'datasets': ['dino_left_large'],
                'bidirectional_lstm': False,
                'lstm_dropout': 0.3,
                'mlp_dropout': 0.3,
                'lr': 0.0002,
                'step_size': 5,
                'gamma': 0.5,
                'weight_decay': 0,
                'hidden_dim': 512,
                'num_layers': 3,
                'batch_size': 64, 
                'frame_frequency': 4,
                'num_epoch': 30,
                'num_workers': 4
                }

device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
print('device: ', device)
set_config_file(config_file, device)
# test_function()

device:  cuda


In [2]:

import warnings

# Suppress the specific UserWarning
warnings.filterwarnings("ignore", category=UserWarning, message=".*copy constructor.*")
warnings.filterwarnings("ignore", category=UserWarning)

In [3]:
from common_functions_python import create_data_loaders, create_model, create_train_dependencies
from tqdm import tqdm
from common_functions_python import get_current_time, plot_result, save_model_result, test_model
from torch.nn.utils.rnn import pack_padded_sequence
import gc
import time

def train_loop():
    train_loader, test_loader = create_data_loaders(config_file['datasets'])

    input_dim = train_loader.dataset[0][0][0].size(0)  # Get input dimension from a single feature from a video
    num_classes = len(set(train_loader.dataset.classes))
    print("input_dim: ", input_dim, " num_classes: ", num_classes)
    print("train_dataset size: ", len(train_loader.dataset))
    print("test_dataset size: ", len(test_loader.dataset))

    model = create_model(input_dim, num_classes)

    criterion, optimizer, scheduler = create_train_dependencies(model)

    print(f"lr {config_file['lr']}, step_size: {config_file['step_size']}, gamma: {config_file['gamma']}, weight_decay: {config_file['weight_decay']}")
    print(f"Model hidden_dim {config_file['hidden_dim']}, num_layers: {config_file['num_layers']}")
    print(f"batch_size {config_file['batch_size']}, frame_frequency: {config_file['frame_frequency']}")

    avg_loss_list = []
    avg_accuracy_list = []
    avg_test_accuracy_list = []
    avg_top5_test_accuracy_list = []
    avg_test_loss_list = []

    num_epoch = config_file['num_epoch']
    for epoch in range(1, num_epoch+1):
        loop = tqdm(train_loader)
        running_loss = 0.0
        running_accuracy= 0.0
        for idx, (features, lengths, labels) in enumerate(loop):
            packed_input = pack_padded_sequence(features, lengths, batch_first=True, enforce_sorted=True)

            # features = features.unsqueeze(-1).float().to(device)
            features = packed_input.to(device)
            lengths = lengths.to(device)
            labels = labels.to(device)

            optimizer.zero_grad()
            outputs = model(features)
            loss = criterion(outputs, labels)

            predictions = outputs.argmax(dim=1, keepdim=True).squeeze()
            correct = (predictions == labels).sum().item()
            accuracy = correct / len(labels)

            loss.backward()
            optimizer.step()

            running_loss += loss.item()
            running_accuracy += 100 * accuracy
            loop.set_description(f"Epoch [{epoch}/{num_epoch}]")
            loop.set_postfix(loss=loss.item(), acc=accuracy)
        scheduler.step()
        avg_loss = running_loss / len(train_loader)
        avg_accuracy = running_accuracy / len(train_loader)
        print(f"Time: {get_current_time()} Epoch [{epoch}], Avg loss: {avg_loss:.4f}, Avg accuracy: {avg_accuracy:.4f}")
        avg_test_accuracy, avg_top5_test_accuracy, avg_test_loss = test_model(test_loader, model, criterion)

        avg_loss_list.append(avg_loss)
        avg_accuracy_list.append(avg_accuracy)
        avg_test_accuracy_list.append(avg_test_accuracy)
        avg_top5_test_accuracy_list.append(avg_top5_test_accuracy)
        avg_test_loss_list.append(avg_test_loss)

    # plot_result(avg_accuracy_list, avg_test_accuracy_list, avg_top5_test_accuracy_list, avg_loss_list, avg_test_loss_list)
    current_time = get_current_time()
    save_model_result(model, current_time, input_dim, num_classes, avg_accuracy_list, avg_test_accuracy_list, avg_top5_test_accuracy_list, avg_loss_list, avg_test_loss_list)

    del train_loader
    del test_loader
    del model
    del criterion
    del optimizer
    del scheduler

In [4]:
import contextlib
import gc

@contextlib.contextmanager
def clear_memory():
    try:
        yield
    finally:
        gc.collect()


datasets_list = [
    ['dino_face_small'],
    ['dino_right_small'],
    # ['deephand_left'],
    # ['dino_left_large'],
    # ['dino_left_small'],
    # ['deephand_left', 'dino_left_large'],
    # ['deephand_left', 'dino_left_large', 'dino_face_small'],
    # ['deephand_left', 'dino_left_large', 'dino_face_small', 'dino_right_small'],
    # ['deephand_left', 'dino_left_small'],
    # ['deephand_left', 'dino_left_small', 'dino_face_small'],
    # ['deephand_left', 'dino_left_small', 'dino_face_small', 'dino_right_small'],
    # ['deephand_left', 'dino_face_small','dino_right_small'],
    # ['dino_left_large', 'dino_face_small', 'dino_right_small'],
    # ['dino_left_small', 'dino_face_small', 'dino_right_small'],
    # ['deephand_left', 'dino_face_small'],
    # ['deephand_left', 'dino_right_small'],
    # ['dino_left_large', 'dino_face_small'],
    # ['dino_left_small', 'dino_face_small'],
    # ['dino_left_large', 'dino_right_small'],
    # ['dino_left_small', 'dino_right_small']
]

dropout_list = [0.2]

frame_frequency_list = [2]

bidirectional_lstm_list = [False]

for datasets in datasets_list:
    for dropout in dropout_list:
        for frame_frequency in frame_frequency_list:
            for bidirectional_lstm in bidirectional_lstm_list:
                gc.collect()
                with clear_memory():   
                    config_file['datasets'] = datasets
                    config_file['lstm_dropout'] = dropout
                    config_file['mlp_dropout'] = dropout
                    config_file['frame_frequency'] = frame_frequency
                    config_file['bidirectional_lstm'] = bidirectional_lstm
                    set_config_file(config_file, device)

                    train_loop()


input_dim:  384  num_classes:  744
train_dataset size:  18018
test_dataset size:  4524
lr 0.0002, step_size: 5, gamma: 0.5, weight_decay: 0
Model hidden_dim 512, num_layers: 3
batch_size 64, frame_frequency: 2


Epoch [1/30]: 100%|██████████| 282/282 [00:06<00:00, 45.19it/s, acc=0.0294, loss=5.81]

Time: 2024-11-23_22-25-39 Epoch [1], Avg loss: 6.2593, Avg accuracy: 0.9080


Accuracy of the network on the 4524 test video: 18.8329 %, top5: 7.6923 %, avg_loss: 0.09113743503361439


Epoch [2/30]: 100%|██████████| 282/282 [00:05<00:00, 47.18it/s, acc=0, loss=5.22]     

...

Time: 2024-11-23_22-32-31 Epoch [29], Avg loss: 1.6005, Avg accuracy: 66.9779


Accuracy of the network on the 4524 test video: 52.2104 %, top5: 62.0027 %, avg_loss: 0.043476538780408025


Epoch [30/30]: 100%|██████████| 282/282 [00:05<00:00, 47.61it/s, acc=0.765, loss=1.26] 

Time: 2024-11-23_22-32-38 Epoch [30], Avg loss: 1.5978, Avg accuracy: 67.0210


Accuracy of the network on the 4524 test video: 52.6967 %, top5: 61.8258 %, avg_loss: 0.04303664601449941
